# Robotic Limb Optimization

This notebook walks through the full Autonomy-IFP-Optimizer workflow on a tubular limb-style part. The focus is the same end-to-end chain: construct the geometry, solve the differentiable IFP route, verify manufacturability, then export robotic path frames and material/process metrics.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from autonomy_ifp_optimizer.config import ExportConfig, LoadCase, OptimizationConfig
from autonomy_ifp_optimizer.core.geometry import load_surface, preview_keepouts, preview_surface, surface_xyz
from autonomy_ifp_optimizer.core.physics import optimize_ifp_path
from autonomy_ifp_optimizer.export.toolpath import compute_metrics, export_kinematics, write_metrics, write_optimized_path
from autonomy_ifp_optimizer.visualize import save_preview


def _set_equal_axes(ax, points):
    mins = points.min(axis=0)
    maxs = points.max(axis=0)
    center = 0.5 * (mins + maxs)
    radius = 0.5 * np.max(maxs - mins)
    radius = max(radius, 1.0e-3)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def plot_surface_geometry(surface, title):
    xyz, faces = preview_surface(surface)
    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, projection="3d")
    mesh = Poly3DCollection(xyz[faces], facecolor="#dde7f0", edgecolor="#90a4ae", linewidths=0.08, alpha=0.35)
    ax.add_collection3d(mesh)
    endpoints = np.asarray(surface_xyz(surface, np.asarray([surface.start_uv, surface.end_uv], dtype=float)))
    ax.scatter(endpoints[:, 0], endpoints[:, 1], endpoints[:, 2], c=["#00bcd4", "#ff9800"], s=55)
    ax.set_title(title)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    _set_equal_axes(ax, np.vstack([xyz, endpoints]))
    plt.show()


def plot_surface_result(surface, result, title, normal_stride=12):
    xyz, faces = preview_surface(surface)
    path_xyz = np.asarray(result["path_xyz"], dtype=float)
    normals = np.asarray(result["normals"], dtype=float)
    control_xyz = np.asarray(surface_xyz(surface, np.asarray(result["control_points_uv"], dtype=float)))
    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, projection="3d")
    mesh = Poly3DCollection(xyz[faces], facecolor="#f4f1ea", edgecolor="#b0bec5", linewidths=0.08, alpha=0.22)
    ax.add_collection3d(mesh)
    ax.plot(path_xyz[:, 0], path_xyz[:, 1], path_xyz[:, 2], color="#1565c0", linewidth=2.6, label="Optimized path")
    ax.plot(control_xyz[:, 0], control_xyz[:, 1], control_xyz[:, 2], "o--", color="#00897b", linewidth=1.5, label="Control polygon")
    ax.quiver(
        path_xyz[::normal_stride, 0],
        path_xyz[::normal_stride, 1],
        path_xyz[::normal_stride, 2],
        normals[::normal_stride, 0],
        normals[::normal_stride, 1],
        normals[::normal_stride, 2],
        length=0.05,
        color="#ff7043",
        normalize=True,
    )
    ax.set_title(title)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.legend(loc="upper right")
    _set_equal_axes(ax, np.vstack([xyz, path_xyz, control_xyz]))
    plt.show()


## 1. Create The Tubular Geometry

This scenario uses the procedural cylinder surface to represent a robotic limb-style part. The route is solved directly on the differentiable surface rather than on a discretized toolpath after the fact.

In [ ]:
surface = load_surface(surface="cylinder")

plot_surface_geometry(surface, "Step 1: tubular limb geometry")
surface.as_dict()


## 2. Define The Structural Demand And IFP Limits

The route is pushed primarily in the axial direction while still respecting the steering-radius and thickness constraints that make the route manufacturable.

In [ ]:
load_case = LoadCase(magnitude_n=650.0, direction_xyz=(0.0, 0.0, 1.0))
optimization_config = OptimizationConfig(
    steps=250,
    min_steering_radius_m=0.05,
    max_thickness=1.10,
)
export_config = ExportConfig(output_dir=repo_root / "outputs" / "robotic_limb_demo")

workflow_inputs = {
    "load_case": {
        "magnitude_n": load_case.magnitude_n,
        "direction_xyz": load_case.direction_xyz,
    },
    "manufacturing_limits": {
        "min_steering_radius_mm": 1000.0 * optimization_config.min_steering_radius_m,
        "max_thickness": optimization_config.max_thickness,
    },
    "output_dir": str(export_config.output_dir),
}
workflow_inputs


## 3. Solve The Differentiable IFP Route

The result contains the optimized Bezier route, normals, objective history, and manufacturability metrics needed for a robotic handoff.

In [ ]:
result = optimize_ifp_path(surface, load_case=load_case, config=optimization_config)
result["metrics"] = compute_metrics(result, export_config)
preview_path = save_preview(result, export_config.output_dir)

plot_surface_result(surface, result, "Step 3: optimized limb path with tool normals")
display(Image(filename=str(preview_path)))

optimization_summary = {
    "objective": round(result["metrics"]["objective"], 4),
    "manufacturable": result["metrics"]["manufacturable"],
    "min_steering_radius_mm": round(result["metrics"]["min_steering_radius_mm"], 2),
    "peak_thickness": round(result["metrics"]["peak_thickness"], 4),
    "path_length_m": round(result["metrics"]["path_length_m"], 4),
    "normalized_compliance": round(result["metrics"]["normalized_compliance"], 4),
    "compliance_n_m": round(result["metrics"]["compliance_n_m"], 6),
    "objective_terms": {key: round(value, 5) for key, value in result["objective_terms"].items()},
}
optimization_summary


## 4. Export Robotic Kinematics

The export stage turns the optimized route into a robotic dataset with XYZ positions plus local normal, tangent, and binormal directions.

In [ ]:
optimized_path_path = write_optimized_path(result, export_config.output_dir)
metrics_path = write_metrics(result["metrics"], export_config.output_dir)
kinematics_json_path = export_kinematics(result, export_config.output_dir, fmt="json")
kinematics_csv_path = export_kinematics(result, export_config.output_dir, fmt="csv")

kinematics_payload = json.loads(kinematics_json_path.read_text(encoding="utf-8"))
artifact_summary = {
    "optimized_path_json": str(optimized_path_path),
    "metrics_json": str(metrics_path),
    "kinematics_json": str(kinematics_json_path),
    "kinematics_csv": str(kinematics_csv_path),
    "preview_png": str(preview_path),
    "sample_robot_records": kinematics_payload["records"][:3],
}
artifact_summary


## 5. Review Material And Process Metrics

These outputs show the manufacturing-facing values that matter after route generation: fiber length, cycle time, thickness level, and overall manufacturability.

In [ ]:
metrics_payload = json.loads(metrics_path.read_text(encoding="utf-8"))
material_and_process_summary = {
    "path_length_m": round(metrics_payload["path_length_m"], 4),
    "toolpath_points": metrics_payload["toolpath_points"],
    "estimated_material_weight_g": round(metrics_payload["estimated_material_weight_g"], 3),
    "estimated_cycle_time_s": round(metrics_payload["estimated_cycle_time_s"], 3),
    "estimated_cycle_time_min": round(metrics_payload["estimated_cycle_time_min"], 3),
    "peak_thickness": round(metrics_payload["peak_thickness"], 4),
    "min_steering_radius_mm": round(metrics_payload["min_steering_radius_mm"], 2),
    "manufacturable": metrics_payload["manufacturable"],
}
material_and_process_summary
